# CPU SESSION STEP 0 — RECURSIVE OFFICIAL EVIDENCE VERIFICATION

## **Scientific purpose**

The initial inventory confirmed that Kaggle mounted the campaign datasets inside a nested **`/kaggle/input/datasets/...`** structure.

This corrected verification therefore discovers the official campaign evidence **recursively**, without assuming that each package exists directly under `/kaggle/input`.

### **Evidence integrity objective**

| Evidence group | Required role |
|---|---|
| **F01–F09** | Closed final campaign evidence |
| **X01** | Frozen Grad-CAM registry and outputs |
| **X02** | Frozen LIME outputs and positive masks |
| **E01** | Frozen 13,711-image F06 external probability evidence |

### **Frozen execution rules**

- **CPU only**
- **No training**
- **No fine-tuning**
- **No new model inference**
- **No external-data download**
- **No analytical metric calculation in Step 0**

<u>The purpose of this step is evidence discovery and verification only.</u>

**Required outcome:** `CPU ANALYSIS READINESS : READY`

In [3]:
from pathlib import Path
import hashlib
import re

import pandas as pd

print("=" * 100)
print("CPU SESSION STEP 0 — RECURSIVE OFFICIAL EVIDENCE VERIFICATION")
print("=" * 100)

INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working/AQUA20_CPU_FINAL")
WORK_ROOT.mkdir(parents=True, exist_ok=True)

# Frozen E01 probability evidence hash
E01_CSV_SHA = "1ab3af604f4ddb56e2f6c99940ffba16ddd4dd7b84ffa73950086046a0aa927b"

AQUA20_CLASSES = [
    "00_coral", "01_crab", "02_diver", "03_eel", "04_fish",
    "05_fishInGroups", "06_flatworm", "07_jellyfish",
    "08_marine_dolphin", "09_octopus", "10_rayfish",
    "11_seaAnemone", "12_seaCucumber", "13_seaSlug",
    "14_seaUrchin", "15_shark", "16_shrimp", "17_squid",
    "18_starfish", "19_turtle",
]

STRICT_EXTERNAL_CLASSES = {
    "Corals", "Crabs", "Dolphin", "Eel", "Fish", "Jelly Fish",
    "Nudibranchs", "Octopus", "Sea Rays", "Sea Urchins",
    "Sharks", "Shrimp", "Squid", "Starfish", "Turtle_Tortoise",
}


def sha256(path):
    """Calculate the integrity digest of frozen scientific evidence."""
    h = hashlib.sha256()

    with open(path, "rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            h.update(chunk)

    return h.hexdigest()


def stage_match(name, stage):
    """Identify an official campaign stage from a mounted directory name."""
    return re.search(
        rf"(^|[^a-z0-9]){stage.lower()}([^0-9]|$)",
        name.lower(),
    ) is not None


def find_file(filename):
    """Locate one frozen evidence file anywhere below Kaggle input."""
    matches = [
        path for path in INPUT_ROOT.rglob(filename)
        if path.is_file()
    ]

    return matches[0] if matches else None


# --------------------------------------------------------------------------------------------------
# Recursively discover mounted campaign directories
# --------------------------------------------------------------------------------------------------

all_dirs = [
    path for path in INPUT_ROOT.rglob("*")
    if path.is_dir()
]

stages = [f"F{i:02d}" for i in range(1, 10)] + ["X01", "X02", "E01"]

stage_candidates = {
    stage: [
        path for path in all_dirs
        if stage_match(path.name, stage)
    ]
    for stage in stages
}

print("\nOFFICIAL CAMPAIGN PACKAGE DISCOVERY")
print("-" * 100)

for stage, candidates in stage_candidates.items():
    status = "FOUND" if candidates else "MISSING"

    print(f"{stage:<8} : {status}")

    # Display one representative mounted path for clean verification
    if candidates:
        print(f"{'':10}{candidates[0]}")


# --------------------------------------------------------------------------------------------------
# Locate the exact frozen scientific evidence
# --------------------------------------------------------------------------------------------------

e01_csv = find_file("E01_all23_external_f06_predictions.csv")
e01_protocol = find_file("E01_external_inference_protocol.csv")
e01_taxonomy = find_file("E01_external_class_taxonomy.csv")
e01_mapping = find_file("E01_strict_shared_class_mapping.csv")
e01_summary = find_file("E01_external_inference_summary.csv")

x01_registry = find_file("X01_frozen_xai_sample_registry.csv")

lime_masks = [
    path for path in INPUT_ROOT.rglob("*")
    if path.is_file()
    and "lime" in [part.lower() for part in path.parts]
    and "masks" in [part.lower() for part in path.parts]
]


# --------------------------------------------------------------------------------------------------
# Verify the frozen E01 probability matrix
# --------------------------------------------------------------------------------------------------

prediction_rows = 0
external_classes = 0
probability_columns = 0
strict_shared_rows = 0
prediction_csv_sha_ok = False
class_col = None
prob_cols = []

if e01_csv is not None:
    df = pd.read_csv(e01_csv)

    class_col = next(
        (
            column for column in [
                "external_class_name",
                "external_class",
                "source_class",
                "folder_name",
                "class_name",
                "true_external_class",
            ]
            if column in df.columns
        ),
        None,
    )

    # Match the exact 20 AQUA20 probability outputs
    for column in df.columns:
        normalized = column.lower()

        for prefix in ("prob_", "p_"):
            if normalized.startswith(prefix):
                normalized = normalized[len(prefix):]

        if normalized in {name.lower() for name in AQUA20_CLASSES}:
            prob_cols.append(column)

    prediction_rows = len(df)
    probability_columns = len(prob_cols)
    prediction_csv_sha_ok = sha256(e01_csv) == E01_CSV_SHA

    if class_col is not None:
        external_classes = df[class_col].nunique()

        strict_shared_rows = int(
            df[class_col]
            .isin(STRICT_EXTERNAL_CLASSES)
            .sum()
        )


# --------------------------------------------------------------------------------------------------
# Frozen readiness protocol
# --------------------------------------------------------------------------------------------------

checks = {
    "F01–F09 official campaign evidence": all(
        stage_candidates[f"F{i:02d}"]
        for i in range(1, 10)
    ),
    "X01 official evidence": bool(stage_candidates["X01"]),
    "X02 official evidence": bool(stage_candidates["X02"]),
    "E01 frozen prediction CSV": e01_csv is not None,
    "E01 prediction CSV SHA-256": prediction_csv_sha_ok,
    "Prediction rows = 13,711": prediction_rows == 13711,
    "External taxonomy classes = 23": external_classes == 23,
    "F06 probability columns = 20": probability_columns == 20,
    "Strict shared images = 9,738": strict_shared_rows == 9738,
    "X01 frozen registry": x01_registry is not None,
    "X02 positive LIME masks = 20": len(lime_masks) == 20,
    "E01 protocol evidence": all(
        path is not None
        for path in [
            e01_protocol,
            e01_taxonomy,
            e01_mapping,
            e01_summary,
        ]
    ),
}


# --------------------------------------------------------------------------------------------------
# Clean paper-oriented evidence summary
# --------------------------------------------------------------------------------------------------

print("\nFROZEN EVIDENCE SUMMARY")
print("-" * 100)

summary_rows = [
    ("E01 prediction CSV", e01_csv or "NOT FOUND"),
    ("External class column", class_col or "NOT FOUND"),
    ("Prediction images", f"{prediction_rows:,}"),
    ("External taxonomy", f"{external_classes} classes"),
    ("F06 output probabilities", f"{probability_columns} columns"),
    ("Strict shared subset", f"{strict_shared_rows:,} images"),
    ("X01 frozen registry", "FOUND" if x01_registry else "NOT FOUND"),
    ("X02 positive LIME masks", f"{len(lime_masks)} masks"),
]

for label, value in summary_rows:
    print(f"{label:<42} : {value}")


print("\nREADINESS VERIFICATION")
print("-" * 100)

for label, passed in checks.items():
    marker = "PASS" if passed else "FAIL"
    print(f"{label:<52} : {marker}")

ready = all(checks.values())


print("\n" + "=" * 100)
print("CPU ANALYSIS READINESS")
print("=" * 100)
print(f"{'Campaign evidence status':<42} : {'READY' if ready else 'NOT READY'}")
print(f"{'Execution mode':<42} : CPU ONLY")
print(f"{'Training / fine-tuning':<42} : NONE")
print(f"{'New external inference':<42} : NONE")
print(f"{'Working root':<42} : {WORK_ROOT}")

print(
    f"{'Next official stage':<42} : "
    f"{'CPU SESSION STEP 1 — E01A RECOMPUTATION' if ready else 'STOP — RESOLVE FAILED EVIDENCE'}"
)

print("=" * 100)

CPU SESSION STEP 0 — RECURSIVE OFFICIAL EVIDENCE VERIFICATION

OFFICIAL CAMPAIGN PACKAGE DISCOVERY
----------------------------------------------------------------------------------------------------
F01      : FOUND
          /kaggle/input/datasets/sazzad1/aqua20-f01-yolo26m-cls-aug200-outputs
F02      : FOUND
          /kaggle/input/datasets/sazzad1/aqua20-f02-yolo26s-cls-aug200-outputs
F03      : FOUND
          /kaggle/input/datasets/sazzad1/aqua20-f03-convnext-small-gentle-cb-focal-outputs
F04      : FOUND
          /kaggle/input/datasets/sazzad1/aqua20-f04-convnext-small-aug200-outputs
F05      : FOUND
          /kaggle/input/datasets/sazzad1/aqua20-f05-convnext-small-aug200-gentle-focal
F06      : FOUND
          /kaggle/input/datasets/sazzad1/aqua20-x02-f06-final-lime-outputs
F07      : FOUND
          /kaggle/input/datasets/sazzad1/aqua20-f07-convnext-small-hr384-tta-outputs
F08      : FOUND
          /kaggle/input/datasets/sazzad1/aqua20-f08-convnext-hr384-yolo26m-prob-ensemb

# CPU SESSION STEP 0B — EXACT PACKAGE AND FROZEN EVIDENCE VERIFICATION

## **Scientific purpose**

The recursive inventory confirmed the presence of the completed **F01–F09**, **X01**, and **X02** campaign evidence.

This correction applies <u>exact stage-prefix matching</u> so that an XAI package containing the text `F06` cannot be misidentified as the official F06 experiment package.

The step also verifies the required **E01 frozen GPU evidence** directly from the saved F06 probability output.

### **Frozen evidence protocol**

| Evidence | Verification role |
|---|---|
| **F01–F09** | Exact official campaign-stage package discovery |
| **X01** | Frozen Grad-CAM registry |
| **X02** | Frozen positive LIME masks |
| **E01** | Frozen external F06 probability evidence |
| **E01 CSV SHA-256** | Scientific evidence integrity |

### **Execution lock**

- **CPU only**
- **No training**
- **No fine-tuning**
- **No new external inference**
- **No external-data download**
- **No E01 metric calculation**

<u>CPU SESSION STEP 1 begins only after every frozen evidence check passes.</u>

In [4]:
from pathlib import Path
import hashlib
import re

import pandas as pd
from IPython.display import display

print("=" * 100)
print("CPU SESSION STEP 0B — EXACT PACKAGE AND FROZEN EVIDENCE VERIFICATION")
print("=" * 100)

INPUT_ROOT = Path("/kaggle/input")
DATASETS_ROOT = INPUT_ROOT / "datasets"
WORK_ROOT = Path("/kaggle/working/AQUA20_CPU_FINAL")
WORK_ROOT.mkdir(parents=True, exist_ok=True)

# Frozen integrity reference for the official E01 probability evidence
E01_CSV_SHA = "1ab3af604f4ddb56e2f6c99940ffba16ddd4dd7b84ffa73950086046a0aa927b"

AQUA20_CLASSES = [
    "00_coral", "01_crab", "02_diver", "03_eel", "04_fish",
    "05_fishInGroups", "06_flatworm", "07_jellyfish",
    "08_marine_dolphin", "09_octopus", "10_rayfish",
    "11_seaAnemone", "12_seaCucumber", "13_seaSlug",
    "14_seaUrchin", "15_shark", "16_shrimp", "17_squid",
    "18_starfish", "19_turtle",
]

STRICT_EXTERNAL_CLASSES = {
    "Corals", "Crabs", "Dolphin", "Eel", "Fish", "Jelly Fish",
    "Nudibranchs", "Octopus", "Sea Rays", "Sea Urchins",
    "Sharks", "Shrimp", "Squid", "Starfish", "Turtle_Tortoise",
}

E01_REQUIRED_FILES = [
    "E01_all23_external_f06_predictions.csv",
    "E01_external_inference_protocol.csv",
    "E01_external_class_taxonomy.csv",
    "E01_strict_shared_class_mapping.csv",
    "E01_external_inference_summary.csv",
    "README_E01_GPU_EVIDENCE.txt",
]


def sha256(path):
    """Verify the integrity of frozen scientific evidence."""
    digest = hashlib.sha256()

    with open(path, "rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)

    return digest.hexdigest()


# Discover the actual Kaggle-mounted dataset directories.
dataset_dirs = []

if DATASETS_ROOT.exists():
    for owner_dir in DATASETS_ROOT.iterdir():
        if owner_dir.is_dir():
            dataset_dirs.extend(
                path for path in owner_dir.iterdir()
                if path.is_dir()
            )

# Exact stage prefixes prevent X02-F06 from being reported as F06.
stage_patterns = {
    **{
        f"F{i:02d}": re.compile(
            rf"^aqua20[-_]f{i:02d}(?:[-_]|$)",
            re.IGNORECASE,
        )
        for i in range(1, 10)
    },
    "X01": re.compile(r"^aqua20[-_]x01(?:[-_]|$)", re.IGNORECASE),
    "X02": re.compile(r"^aqua20[-_]x02(?:[-_]|$)", re.IGNORECASE),
    "E01": re.compile(r"^aqua20[-_]e01(?:[-_]|$)", re.IGNORECASE),
}

stage_candidates = {
    stage: [
        path for path in dataset_dirs
        if pattern.search(path.name)
    ]
    for stage, pattern in stage_patterns.items()
}

package_rows = []

for stage, candidates in stage_candidates.items():
    package_rows.append({
        "Stage": stage,
        "Status": "FOUND" if candidates else "MISSING",
        "Mounted package": candidates[0].name if candidates else "—",
    })

print("\nOFFICIAL CAMPAIGN PACKAGE TABLE")
print("-" * 100)

package_table = pd.DataFrame(package_rows)
display(package_table)


# Build one case-insensitive evidence index from all mounted files.
all_files = [
    path for path in INPUT_ROOT.rglob("*")
    if path.is_file()
]

file_index = {}

for path in all_files:
    file_index.setdefault(path.name.lower(), []).append(path)


def find_ci(filename):
    """Locate an official filename without depending on the Kaggle dataset slug."""
    matches = file_index.get(filename.lower(), [])
    return matches[0] if matches else None


e01_files = {
    filename: find_ci(filename)
    for filename in E01_REQUIRED_FILES
}

e01_csv = e01_files["E01_all23_external_f06_predictions.csv"]
x01_registry = find_ci("X01_frozen_xai_sample_registry.csv")

lime_masks = [
    path for path in all_files
    if "lime" in [part.lower() for part in path.parts]
    and "masks" in [part.lower() for part in path.parts]
]


# Verify the frozen 13,711 × 20 E01 probability evidence.
prediction_rows = 0
external_classes = 0
probability_columns = 0
strict_shared_rows = 0
class_col = None
csv_sha_ok = False

if e01_csv is not None:
    df = pd.read_csv(e01_csv)

    lower_columns = {
        column.lower(): column
        for column in df.columns
    }

    class_col = next(
        (
            lower_columns[name]
            for name in [
                "external_class_name",
                "external_class",
                "source_class",
                "folder_name",
                "class_name",
                "true_external_class",
            ]
            if name in lower_columns
        ),
        None,
    )

    class_names_lower = {
        name.lower()
        for name in AQUA20_CLASSES
    }

    prob_cols = []

    for column in df.columns:
        normalized = column.lower()

        if normalized.startswith("prob_"):
            normalized = normalized[5:]
        elif normalized.startswith("p_"):
            normalized = normalized[2:]

        if normalized in class_names_lower:
            prob_cols.append(column)

    prediction_rows = len(df)
    probability_columns = len(prob_cols)
    csv_sha_ok = sha256(e01_csv) == E01_CSV_SHA

    if class_col is not None:
        external_classes = df[class_col].nunique()

        strict_shared_rows = int(
            df[class_col]
            .isin(STRICT_EXTERNAL_CLASSES)
            .sum()
        )


checks = {
    "F01–F09 exact campaign packages": all(
        stage_candidates[f"F{i:02d}"]
        for i in range(1, 10)
    ),
    "X01 exact official package": bool(stage_candidates["X01"]),
    "X02 exact official package": bool(stage_candidates["X02"]),
    "E01 exact official package": bool(stage_candidates["E01"]),
    "E01 six required evidence files": all(e01_files.values()),
    "E01 prediction CSV SHA-256": csv_sha_ok,
    "Prediction images = 13,711": prediction_rows == 13711,
    "External taxonomy = 23 classes": external_classes == 23,
    "F06 probabilities = 20 columns": probability_columns == 20,
    "Strict shared subset = 9,738": strict_shared_rows == 9738,
    "X01 frozen registry": x01_registry is not None,
    "X02 positive LIME masks = 20": len(lime_masks) == 20,
}

print("\nFROZEN EVIDENCE SUMMARY")
print("-" * 100)

print(f"{'E01 prediction CSV':<42} : {e01_csv or 'NOT FOUND'}")
print(f"{'External class column':<42} : {class_col or 'NOT FOUND'}")
print(f"{'Prediction images':<42} : {prediction_rows:,}")
print(f"{'External taxonomy':<42} : {external_classes} classes")
print(f"{'F06 probability outputs':<42} : {probability_columns} columns")
print(f"{'Strict shared subset':<42} : {strict_shared_rows:,} images")
print(f"{'X01 frozen registry':<42} : {'FOUND' if x01_registry else 'NOT FOUND'}")
print(f"{'X02 positive LIME masks':<42} : {len(lime_masks)}")

print("\nREADINESS VERIFICATION")
print("-" * 100)

for label, passed in checks.items():
    print(f"{label:<52} : {'PASS' if passed else 'FAIL'}")

ready = all(checks.values())

print("\n" + "=" * 100)
print("CPU ANALYSIS READINESS")
print("=" * 100)
print(f"{'Campaign evidence status':<42} : {'READY' if ready else 'NOT READY'}")
print(f"{'Execution mode':<42} : CPU ONLY")
print(f"{'Training / fine-tuning':<42} : NONE")
print(f"{'New external inference':<42} : NONE")
print(f"{'Working root':<42} : {WORK_ROOT}")
print(
    f"{'Next official stage':<42} : "
    f"{'CPU SESSION STEP 1 — E01A RECOMPUTATION' if ready else 'STOP — RESOLVE FAILED EVIDENCE'}"
)
print("=" * 100)

CPU SESSION STEP 0B — EXACT PACKAGE AND FROZEN EVIDENCE VERIFICATION

OFFICIAL CAMPAIGN PACKAGE TABLE
----------------------------------------------------------------------------------------------------


,Stage,Status,Mounted package
0,F01,FOUND,aqua20-f01-yolo26m-cls-aug200-outputs
1,F02,FOUND,aqua20-f02-yolo26s-cls-aug200-outputs
2,F03,FOUND,aqua20-f03-convnext-small-gentle-cb-focal-outputs
3,F04,FOUND,aqua20-f04-convnext-small-aug200-outputs
4,F05,FOUND,aqua20-f05-convnext-small-aug200-gentle-focal
5,F06,FOUND,aqua20-f06-convnext-small-hr384-gentle-focal
6,F07,FOUND,aqua20-f07-convnext-small-hr384-tta-outputs
7,F08,FOUND,aqua20-f08-convnext-hr384-yolo26m-prob-ensemble
8,F09,FOUND,aqua20-f09-final-method-freeze-campaign-outputs
9,X01,FOUND,aqua20-x01-f06-final-gradcam-outputs



FROZEN EVIDENCE SUMMARY
----------------------------------------------------------------------------------------------------
E01 prediction CSV                         : NOT FOUND
External class column                      : NOT FOUND
Prediction images                          : 0
External taxonomy                          : 0 classes
F06 probability outputs                    : 0 columns
Strict shared subset                       : 0 images
X01 frozen registry                        : FOUND
X02 positive LIME masks                    : 20

READINESS VERIFICATION
----------------------------------------------------------------------------------------------------
F01–F09 exact campaign packages                      : PASS
X01 exact official package                           : PASS
X02 exact official package                           : PASS
E01 exact official package                           : FAIL
E01 six required evidence files                      : FAIL
E01 prediction CSV SHA-256   